# 05 - Silver: TRM (Tasa de Cambio Representativa del Mercado)

Trabajo Práctico - Pipeline ETL end-to-end (Bronze → Silver / DQL → Gold)

| | Tabla |
|---|---|
| Entrada | `datos_empresas.bronze.de_semiestructurado_tasacambio_api` |
| Salida Silver | `datos_empresas.silver.de_tasacambio` |
| Salida DQL | `datos_empresas.dql.de_tasacambio_dql` |
| Llave (grano) | `fecha_vigencia_desde` (una TRM por periodo de vigencia) |

## Por qué entra TRM al pipeline

TRM no tiene una llave común con RUES ni con SECOP, pero se cruza **por fecha**. En Gold sirve para dos cosas:

* Convertir el valor de cada contrato a dólares con la TRM vigente el día de la firma (`gold.de_fact_contratos`).
* Comparar la TRM promedio mensual con la creación y la cancelación de empresas (`gold.de_agg_indicadores_mes`).

## Transformaciones

| Bronze | Silver | Transformación |
|---|---|---|
| `valor` | `valor_trm` | texto → `DECIMAL(12,2)` |
| `vigenciadesde` | `fecha_vigencia_desde` | texto ISO → `DATE` |
| `vigenciahasta` | `fecha_vigencia_hasta` | texto ISO → `DATE` (un viernes vale hasta el lunes) |
| `unidad` | `unidad` | `trim` + mayúsculas |

## Reglas de calidad (rechazo hacia DQL)

| # | Regla (`_motivo_rechazo`) | Condición | Por qué es pertinente |
|---|---|---|---|
| 1 | `valor_trm_fuera_de_rango` | Valor nulo, no numérico o fuera de 1.000–10.000 COP por dólar | La TRM histórica ha estado entre unos 1.700 y 5.100 COP. Un valor fuera del rango indica un error de unidad o de carga (por ejemplo, decimales corridos) y produciría conversiones a dólares absurdas en Gold. |
| 2 | `vigencia_invalida` | Falta alguna fecha de vigencia o `vigenciadesde > vigenciahasta` | El cruce con los contratos busca la TRM cuyo rango contiene la fecha de firma. Un rango invertido o incompleto nunca coincidiría, o coincidiría con la fecha equivocada. |
| 3 | `unidad_invalida` | `unidad` distinta de `COP` | Toda la conversión supone pesos colombianos por dólar. Otra unidad haría que los valores no fueran comparables. |
| 4 | `duplicado_version_anterior` | Hay una carga más reciente para la misma `fecha_vigencia_desde` | **Unicidad**: una sola TRM por día de vigencia. Si hubiera dos, cada contrato se duplicaría al cruzarlo en Gold. |

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: Lectura de Bronze y tipado

In [ ]:
%python
df_bronze = preparar_bronze(spark.table(BRONZE_TRM))

df_tipado = df_bronze.select(
    "_registro_bronze", "_hash_bronze", "_id_registro_bronze",
    F.col("_ingested_at").alias("_bronze_ingested_at"),
    fecha_iso("vigenciadesde").alias("fecha_vigencia_desde"),
    fecha_iso("vigenciahasta").alias("fecha_vigencia_hasta"),
    decimal_seguro("valor", "DECIMAL(12,2)").alias("valor_trm"),
    texto_limpio("unidad", mayusculas=True).alias("unidad"),
)

df_versionado = marcar_version(df_tipado, ["fecha_vigencia_desde"], [F.col("_bronze_ingested_at").desc()])

---
## Paso 1.5: Diagnóstico previo — evidencia detrás de cada regla

Resultado con el Bronze del 21-oct-2026 (235 filas, del 2025-09-19 al 2026-09-19): 0 en todas las reglas. La TRM es una serie oficial y llega limpia; las reglas quedan como salvaguarda para cargas futuras.

In [ ]:
%sql
-- Evidencia: filas de Bronze que incumplirían cada regla (mismas condiciones que el Paso 2)
WITH t AS (
  SELECT *,
         TRY_CAST(valor AS DECIMAL(12,2)) AS v,
         TO_DATE(TRY_TO_TIMESTAMP(vigenciadesde, "yyyy-MM-dd'T'HH:mm:ss.SSS")) AS desde,
         TO_DATE(TRY_TO_TIMESTAMP(vigenciahasta, "yyyy-MM-dd'T'HH:mm:ss.SSS")) AS hasta
  FROM datos_empresas.bronze.de_semiestructurado_tasacambio_api
),
v AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY desde ORDER BY _ingested_at DESC) AS version_fila FROM t
)
SELECT
  COUNT(*) AS filas_bronze,
  SUM(CASE WHEN v IS NULL OR v NOT BETWEEN 1000 AND 10000 THEN 1 ELSE 0 END) AS r1_valor_trm_fuera_de_rango,
  SUM(CASE WHEN desde IS NULL OR hasta IS NULL OR desde > hasta THEN 1 ELSE 0 END) AS r2_vigencia_invalida,
  SUM(CASE WHEN UPPER(TRIM(COALESCE(unidad, ''))) <> 'COP' THEN 1 ELSE 0 END) AS r3_unidad_invalida,
  SUM(CASE WHEN desde IS NOT NULL AND version_fila > 1 THEN 1 ELSE 0 END) AS r4_duplicado_version_anterior
FROM v;

---
## Paso 2: Reglas de calidad y separación Silver / DQL

In [ ]:
%python
REGLAS = [
    ("valor_trm_fuera_de_rango",
     F.col("valor_trm").isNull() | ~F.col("valor_trm").between(1000, 10000)),
    ("vigencia_invalida",
     F.col("fecha_vigencia_desde").isNull() | F.col("fecha_vigencia_hasta").isNull()
     | (F.col("fecha_vigencia_desde") > F.col("fecha_vigencia_hasta"))),
    ("unidad_invalida",
     ~F.coalesce(F.col("unidad") == "COP", F.lit(False))),
    ("duplicado_version_anterior",
     (F.col("_version") > 1) & F.col("fecha_vigencia_desde").isNotNull()),
]

df_aceptados, df_rechazados = separar_aceptados_rechazados(df_versionado, REGLAS)

COLUMNAS_SILVER = ["fecha_vigencia_desde", "fecha_vigencia_hasta", "valor_trm", "unidad"]

df_silver = construir_silver(df_aceptados, COLUMNAS_SILVER)
df_dql = construir_dql(df_rechazados)

---
## Paso 3: Escritura idempotente (MERGE) en Silver y DQL

In [ ]:
%python
m_silver = escribir_merge(
    df_silver, SILVER_TRM, ["fecha_vigencia_desde"], capa="silver",
    comentario="Silver - TRM diaria validada (COP por USD), una fila por fecha de vigencia.",
)
m_dql = escribir_merge(
    df_dql, DQL_TRM, ["_id_registro_bronze"], capa="dql", columna_fecha="_fecha_rechazo",
    comentario="DQL - Registros de TRM rechazados, con la fila original de Bronze y el motivo.",
)

conteos = conciliar_fuente("TRM")

---
## Paso 4: Validación

In [ ]:
%sql
SELECT COUNT(*) AS dias, MIN(fecha_vigencia_desde) AS desde, MAX(fecha_vigencia_hasta) AS hasta,
       MIN(valor_trm) AS trm_minima, MAX(valor_trm) AS trm_maxima
FROM datos_empresas.silver.de_tasacambio;

In [ ]:
%sql
SELECT _motivo_rechazo, COUNT(*) AS registros
FROM datos_empresas.dql.de_tasacambio_dql
GROUP BY _motivo_rechazo
ORDER BY registros DESC;

In [ ]:
%python
finalizar({"notebook": "05_Silver_TRM", "conteos": conteos, "silver": m_silver, "dql": m_dql})